In [19]:
!git clone https://github.com/AIVIETNAM-AIO-ERICPHAM73/hydraulic-machine-health.git

fatal: destination path 'hydraulic-machine-health' already exists and is not an empty directory.


In [24]:
%cd /content/hydraulic-machine-health

!git init

!git config --global user.email "pvnghi9@gmail.com"
!git config --global user.name "EricPham7395"

!git branch -M main

!git remote set-url origin https://github.com/AIVIETNAM-AIO-ERICPHAM73/hydraulic-machine-health.git

!git add .
# Commit changes with a message
!git commit -m "update"

# Configure rebase strategy for pull
!git config pull.rebase true

# Pull changes from GitHub before pushing
!git pull origin main

# Push changes to GitHub
!git push origin main

/content/hydraulic-machine-health
Reinitialized existing Git repository in /content/hydraulic-machine-health/.git/
On branch main
Your branch is ahead of 'origin/main' by 1 commit.
  (use "git push" to publish your local commits)

nothing to commit, working tree clean
remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 4 (delta 3), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 1.26 KiB | 646.00 KiB/s, done.
From https://github.com/AIVIETNAM-AIO-ERICPHAM73/hydraulic-machine-health
 * branch            main       -> FETCH_HEAD
   725d516..29503ee  main       -> origin/main
Successfully rebased and updated refs/heads/main.
fatal: could not read Username for 'https://github.com': No such device or address


In [4]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [42]:
from pathlib import Path
ROOT_DRIVE_DIR = Path('/content/drive/MyDrive/AIO/AIO_Module3_Hydraulic')
RAW_DATA_DRIVE_DIR = ROOT_DRIVE_DIR / 'data_raw'
PROCESSED_DATA_DRIVE_DIR = ROOT_DRIVE_DIR / 'data_processed'
ARTIFACTS_DRIVE_DIR = ROOT_DRIVE_DIR / 'artifacts'
for p in [RAW_DATA_DRIVE_DIR, PROCESSED_DATA_DRIVE_DIR, ARTIFACTS_DRIVE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

In [43]:
#%%writefile hydraulic-machine-health/src/config.py
from pathlib import Path
from typing import Final

import sys, os

# ---------------------------------------------------------------------------
# Reproducibility
# ---------------------------------------------------------------------------

RANDOM_STATE: Final[int] = 42

# ---------------------------------------------------------------------------
# Dataset contract
# ---------------------------------------------------------------------------

EXPECTED_CYCLES: Final[int] = 2205
CYCLE_DURATION_SECONDS: Final[int] = 60

PROFILE_COLUMNS: Final[list[str]] = ['cooler_condition', 'valve_condition', 'internal_pump_leakage', 'hydraulic_accumulator', 'stable_flag']

TARGET_COLUMN: Final[str] = 'pump_leakage'
META_COLUMNS: Final[list[str]] = ['cooler_condition', 'valve_condition', 'hydraulic_accumulator', 'stable_flag']

PUMP_LABELS: Final[dict[int, str]] = {0: 'no_leakage', 1: 'weak_leakage', 2: 'severe_leakage'}

EXCLUDED_MODEL_COLUMNS: Final[list[str]] = ['cooler_condition', 'valve_condition', 'hydraulic_accumulator', 'stable_flag', 'cycle_id']

# ---------------------------------------------------------------------------
# Sensor metadata
# ---------------------------------------------------------------------------

SENSORS: Final[list[str]] = [
    'PS1', 'PS2', 'PS3', 'PS4', 'PS5', 'PS6',
    'EPS1',
    'FS1', 'FS2',
    'TS1', 'TS2', 'TS3', 'TS4',
    'VS1',
    'CE', 'CP', 'SE'
]

SENSOR_SAMPLING_HZ: Final[dict[str, int]] = {
    'PS1': 100, 'PS2': 100, 'PS3': 100, 'PS4': 100, 'PS5': 100, 'PS6': 100,
    'EPS1': 100,
    'FS1': 10, 'FS2': 10,
    'TS1': 1, 'TS2': 1, 'TS3': 1, 'TS4': 1,
    'VS1': 1,
    'CE': 1, 'CP': 1, 'SE': 1
}

# Expected time points per cycle = sampling rate(Hz) * 60(seconds)
EXPECTED_TIMEPOINTS: Final[dict[str, int]] = {
    sensor: hz * CYCLE_DURATION_SECONDS for sensor, hz in SENSOR_SAMPLING_HZ.items()
}


# ---------------------------------------------------------------------------
# Feature engineering configuration
# ---------------------------------------------------------------------------

START_END_FRACTION: Final[float] = 0.10
N_SEGMENTS_V2: Final[int] = 6


# ---------------------------------------------------------------------------
# Validation / research configuration
# ---------------------------------------------------------------------------

N_SPLITS: Final[int] = 5
BLOCKED_HOLDOUT_FRACTION: Final[float] = 0.20
NOISE_SNR_DB_LEVELS: Final[tuple[int, ...]] = (30, 20, 10)
SHAP_TOP_K: Final[int] = 10


# ---------------------------------------------------------------------------
# Path convention
# ---------------------------------------------------------------------------

def get_project_root() -> Path:
    # Check for running on Colab or VS Code
    if 'google.colab' in sys.modules:
        # If Colab
        project_root_path = Path('/content/hydraulic-machine-health')
    elif '__vsc_ipynb_file__' in globals():
        # If VS Code
        notebook_path = Path(globals()['__vsc_ipynb_file__'])
        project_root_path = notebook_path.parent.parent
    else:
        # Backup for normal Jupyter Notebook or file.py
        try:
            notebook_path = Path(__file__).resolve()
            project_root_path = notebook_path.parent.parent
        except NameError:
            # If cannot found, get the current working folder
            project_root_path = Path(os.getcwd())

    # Trace back to the root folder and config sys.path
    os.chdir(project_root_path)

    if str(project_root_path) not in sys.path:
        sys.path.append(str(project_root_path))

    return project_root_path


In [41]:
#%%writefile hydraulic-machine-health/src/data.py
from pathlib import Path
from typing import Iterable

import numpy as np
import pandas as pd

root_dir = get_project_root()

from src.config import (EXPECTED_CYCLES, EXPECTED_TIMEPOINTS, META_COLUMNS, PROFILE_COLUMNS, TARGET_COLUMN, SENSORS)

def load_profile(profile_path: Path) -> pd.DataFrame:
    # Read profile.txt as a tab-delimited file with no header
    profile_arr = np.loadtxt(profile_path, delimiter='\t', dtype=np.int64)

    # Assign PROFILE_COLUMNS in the documented order
    profile_df = pd.DataFrame(profile_arr, columns=PROFILE_COLUMNS, dtype=np.int64)

    # Create a cycle_id index from row order
    profile_df['cycle_id'] = profile_df.index

    # Bring cycle_id column to first column
    cycle_id_col = profile_df.pop('cycle_id')
    profile_df.insert(0, 'cycle_id', cycle_id_col)

    # Preserve all five profile columns for audit and sensitivity analysis
    return profile_df


def validate_profile(profile_df: pd.DataFrame, expected_cycles: int = EXPECTED_CYCLES) -> dict[str, object]:
    # Validate row count equals expected_cycles
    if profile_df.shape[0] != expected_cycles:
        print(f'Profile row counts need exacty match {expected_cycles}, but got {profile_df.shape[0]}, recheck again!')
    else:
        is_row_count_valid = True

    # Validate required profile columns exist
    required_columns = set('cooler_condition', 'valve_condition', 'internal_pump_leakage', 'hydraulic_accumulator', 'stable_flag')
    if not required_columns.issubset(profile_df.columns):
        missing_columns = required_columns - set(profile_df.columns)
        print(f'Profile required missing columns: {missing_columns}')
    else:
        is_required_columns_exist = True

    # Validate pump_leakage values are a subset of {0, 1, 2}
    pump_leakage_values = set(profile_df[TARGET_COLUMN].unique())
    if not set([0, 1, 2]).issubset(pump_leakage_values):
        print(f'Profile pump_leakage values need to be a subset of {0, 1, 2}, but got {pump_leakage_values}, recheck again!')
    else:
        is_pump_leakage_valid = True

    # Validate stable_flag values are subset of {0, 1}
    stable_flag_values = set(profile_df['stable_flag'].unique())
    if not set([0, 1]).issubset(stable_flag_values):
        print(f'Profile stable_flag values need to be a subset of {0, 1}, but got {stable_flag_values}, recheck again!')
    else:
        is_stable_flag_valid = True

    # Check duplicated cycle_id values
    if profile_df['cycle_id'].duplicated().any():
        print(f'Profile duplicated cycle_id values found, cycle_id need unique, recheck again!')
    else:
        is_cycle_id_unique = True

    # Check missing values in profile metadata
    if profile_df[META_COLUMNS].isna().values.any():
        print(f'Profile metadata missing values found, recheck again!')
    else:
        is_meta_columns_valid = True

    return {
        'n_cycles': profile_df.shape[0],
        'missing_counts': profile_df.isna().sum(),
        'target_values': profile_df[TARGET_COLUMN].unique(),
        'stable_flag_values': profile_df['stable_flag'].unique(),
        'is_valid': all([is_row_count_valid,
                         is_required_columns_exist,
                         is_pump_leakage_valid,
                         is_stable_flag_valid,
                         is_cycle_id_unique,
                         is_meta_columns_valid])
    }


def load_sensor(sensor_path: Path, dtype: np.dtype = np.float32) -> np.ndarray:
    # Read one by one tab-delimited sensor file into a 2D NumPy array.
    sensor_arr = np.loadtxt(sensor_path, delimiter='\t', dtype=dtype)

    return sensor_arr


def validate_sensor_array(sensor_arr: np.ndarray, sensor_name: str, expected_cycles: int = EXPECTED_CYCLES) -> dict[str, object]:
    # Validate number of dimensions of sensor array, expected = 2
    if sensor_arr.ndim != 2:
        print(f'Dimensions of sensor array is not equal to 2, recheck again!')
    else:
        is_ndim_valid = True

    # Validate row count equals expected_cycles
    if sensor_arr.shape[0] != expected_cycles:
        print(f'Sensor {sensor_name} row counts need exacty match {expected_cycles}, but got {sensor_arr.shape[0]}, recheck again!')
    else:
        is_row_count_valid = True

    # Validate column count equals expected_timepoints of each sensor_name
    if sensor_arr.shape[1] != EXPECTED_TIMEPOINTS[sensor_name]:
        print(f'Sensor {sensor_name} column counts need exacty match {EXPECTED_TIMEPOINTS[sensor_name]}, but got {sensor_arr.shape[1]}, recheck again!')
    else:
        is_column_count_valid = True

    # Check NaN (Not a Number) or Inf (Infinity) counts
    if np.isnan(sensor_arr).any() or np.isinf(sensor_arr).any():
        print(f'Sensor {sensor_name} contains NaN or Inf values, recheck again!')
    else:
        is_nan_inf_valid = True


    return {
        'sensor': sensor_name,
        'shape': sensor_arr.shape,
        'nan_count': np.isnan(sensor_arr).sum(),
        'inf_count': np.isinf(sensor_arr).sum(),
        'is_valid': all([is_ndim_valid, is_row_count_valid, is_column_count_valid, is_nan_inf_valid])
    }


def get_sensor_paths(raw_data_dir: Path, sensors: Iterable[str]) -> dict[str, Path]:
    # Build mapping and validate each file exists before feature extraction begins
    sensor_paths = {sensor: raw_data_dir / f'{sensor}.txt' for sensor in sensors if (raw_data_dir / f'{sensor}.txt').exists()}

    required_sensors = set(SENSORS)

    if not required_sensors.issubset(sensor_paths.keys()):
        missing_sensors = required_sensors - set(sensor_paths.keys())
        print(f'Missing sensors: {missing_sensors}')

    return sensor_paths


def load_engineered_dataset(features_path: Path, profile_path: Path) -> tuple[pd.DataFrame, pd.Series, pd.DataFrame]:
    # Load engineered feature table from Parquet
    features_df = pd.read_parquet(features_path)

    # Load profile metadata
    profile_df = load_profile(profile_path)

    # Align rows strictly by cycle_id
    features_df = features_df.set_index('cycle_id').loc[profile_df['cycle_id']].reset_index()

    # Build X dataframe from engineered features only
    X = features_df.drop(columns=['cycle_id'])

    # Build y dataframe from TARGET_COLUMN
    y = profile_df[TARGET_COLUMN]

    # Build meta dataframe from (META_COLUMNS + cycle_id) for audit
    meta = profile_df[META_COLUMNS + ['cycle_id']]

    return X, y, meta


def save_dataframe(df: pd.DataFrame, output_path: Path) -> None:
    # Create parent directory if needed
    if not output_path.parent.exists():
        output_path.parent.mkdir(parents=True, exist_ok=True)

    # Save Parquet for processed feature tables
    df.index.name = 'cycle_id'
    df.to_parquet(output_path, index=True)

    return None

In [45]:
#%%writefile hydraulic-machine-health/src/features.py
from pathlib import Path
from typing import Iterable

import gc
import numpy as np
import pandas as pd

root_dir = get_project_root()

from src.data import load_sensor, validate_sensor_array

def extract_cycle_features(sensor_arr: np.ndarray, prefix: str, start_end_fraction: float = START_END_FRACTION) -> pd.DataFrame:
    # Compute one row of engineered features per cycle of sensor array (n_cycles, n_timepoints)
    # Core features
    mean_sensor = sensor_arr.mean(axis=1)
    std_sensor = sensor_arr.std(axis=1)
    min_sensor = sensor_arr.min(axis=1)
    max_sensor = sensor_arr.max(axis=1)
    range_sensor = max_sensor - min_sensor
    median_sensor = np.median(sensor_arr, axis=1)

    q25_sensor = np.quantile(sensor_arr, 0.25, axis=1)
    q75_sensor = np.quantile(sensor_arr, 0.75, axis=1)
    iqr_sensor = q75_sensor - q25_sensor

    k = max(1, int(start_end_fraction * sensor_arr.shape[1]))
    start_mean_sensor = sensor_arr[:, :k].mean(axis=1)
    end_mean_sensor = sensor_arr[:, -k:].mean(axis=1)
    delta_sensor = end_mean_sensor - start_mean_sensor

    # Linear trend / slope for each cycle
    x = np.arange(sensor_arr.shape[1], dtype=np.float32)
    x_center = x - x.mean()
    denominator =  np.sum(x_center**2)
    slope_sensor = np.dot(sensor_arr, x_center) / denominator

    return pd.DataFrame({
        f'{prefix}_mean': mean_sensor,
        f'{prefix}_std': std_sensor,
        f'{prefix}_min': min_sensor,
        f'{prefix}_max': max_sensor,
        f'{prefix}_range': range_sensor,
        f'{prefix}_median': median_sensor,
        f'{prefix}_q25': q25_sensor,
        f'{prefix}_q75': q75_sensor,
        f'{prefix}_iqr': iqr_sensor,
        f'{prefix}_start_mean': start_mean_sensor,
        f'{prefix}_end_mean': end_mean_sensor,
        f'{prefix}_delta': delta_sensor,
        f'{prefix}_slope': slope_sensor
    })

def extract_segment_features(sensor_arr: np.ndarray, prefix: str, n_segments: int = N_SEGMENTS_V2) -> pd.DataFrame:
    # For Version 2

    # Divide each 60-second cycle into n_segments contiguous segments
    # Default n_segments=6, each segment represents approximately 10 seconds
    sensor_arr_segments = np.array_split(sensor_arr, n_segments, axis=1)

    segments_features = []
    for idx, segment in enumerate(sensor_arr_segments):
        segment_features = extract_cycle_features(segment, prefix + f'_seg{idx}')
        segments_features.append(segment_features)

    return pd.concat(segments_features, axis=1)


def extract_sensor_feature_table(sensor_path: Path, sensor_name: str, include_segment_v2: bool = False) -> pd.DataFrame:
    # Load one sensor array
    sensor_arr = load_sensor(sensor_path)

    # Validate sensor array shape
    sensor_validation = validate_sensor_array(sensor_arr, sensor_name)
    if sensor_validation['is_valid']:
        print(f'=== Sensor {sensor_name} is valid! Shape: {sensor_validation['shape']} ===')
    else:
        raise ValueError(f'Sensor {sensor_name} is not valid!')

    # Extract v1 global cycle features
    global_cycle_features = extract_cycle_features(sensor_arr, sensor_name)

    # Extract v2 segment features
    if include_segment_v2:
        segment_features = extract_segment_features(sensor_arr, sensor_name)

        # Release the raw sensor array after extraction to control memory use
        del sensor_arr
        gc.collect()

        return pd.concat([global_cycle_features, segment_features], axis=1)

    # Release the raw sensor array after extraction to control memory use
    del sensor_arr
    gc.collect()

    return global_cycle_features


def build_feature_table(raw_dir: Path, sensors: Iterable[str] = SENSORS, include_segment_v2: bool = False) -> pd.DataFrame:
    # Get sensor paths
    sensor_paths = get_sensor_paths(raw_dir, sensors)

    # Iterate sensor-by-sensor and get sensor_feature_tables
    sensor_feature_tables = []
    for sensor_name, sensor_path in sensor_paths.items():
        sensor_feature_table = extract_sensor_feature_table(sensor_path, sensor_name, include_segment_v2=include_segment_v2)
        sensor_feature_tables.append(sensor_feature_table)

    # Concatenate all sensor feature DataFrames column-wise into one DataFrame
    X_features = pd.concat(sensor_feature_tables, axis=1)
    X_features.index.name = 'cycle_id'

    # Validate final row count equals EXPECTED_CYCLES
    if X_features.shape[0] != EXPECTED_CYCLES:
        print(f'Feature table row counts need exacty match {EXPECTED_CYCLES}, but got {X_features.shape[0]}, recheck again!')
    else:
        print(f'Feature table row counts match {EXPECTED_CYCLES}')

    return X_features


# def build_feature_dictionary(feature_columns: Iterable[str]) -> pd.DataFrame:
#     # Parse each feature name into sensor prefix and statistic/feature type
#     prefixes, sensors, feature_types, definitions, formulas = [], [], [], [], []


#     for feature_column in feature_columns:
#         if feature_column == 'cycle_id':
#             continue
#         if feature_column.split('_')[1] == 'seg':
#             prefixes.append(feature_column.split('_')[0])
#             feature_types.append(feature_column.split('_')[1] + '_' + feature_column.split('_')[2])
#         else:
#             prefixes.append(feature_column.split('_')[0])
#             feature_types.append(feature_column.split('_')[1])

#         if set('PS1', 'PS2', 'PS3', 'PS4', 'PS5', 'PS6').issubset(feature_column.split('_')[0]):
#                 sensors.append('Pressure')
#                 definitions.append('Pressure in bar')
#         elif set('EPS1').issubset(feature_column.split('_')[0]):
#                 sensors.append('Motor Power')
#         elif set('FS1', 'FS2').issubset(feature_column.split('_')[0]):
#                 sensors.append('Volume Flow')
#         elif set('TS1', 'TS2', 'TS3', 'TS4').issubset(feature_column.split('_')[0]):
#                 sensors.append('Temperature')
#         elif set('VS1').issubset(feature_column.split('_')[0]):
#                 sensors.append('Vibration')
#         elif set('CE').issubset(feature_column.split('_')[0]):
#                 sensors.append('Cooling Efficiency (Virtual)')
#         elif set('CP').issubset(feature_column.split('_')[0]):
#                 sensors.append('Cooling Power (Virtual)')
#         elif set('SE').issubset(feature_column.split('_')[0]):
#                 sensors.append('Efficiency Factor')


def audit_engineered_features(X: pd.DataFrame, high_corr_threshold: float = 0.98) -> dict[str, object]:
    # Count NaN and inf values:
    nan_count = X.isna().sum()
    inf_count = X.isin([np.inf, -np.inf]).sum()

    # Identify constant / near-constant columns
    constant_columns = [col for col in X.columns if X[col].nunique() <= 1]
    near_constant_columns = [col for col in X.columns if X[col].nunique() >= 2 and X[col].value_counts(normalize=True).max() > 0.95]

    # Summarize duplicate column names
    duplicate_columns = X.columns[X.columns.duplicated()]

    # Identify highly correlated feature pairs for interpretation only
    corr_matrix = X.corr().abs()
    upper_triangle = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    highly_correlated_pairs = [(col1, col2) for col1, col2 in zip(*np.where(upper_triangle > high_corr_threshold)) if col1 != col2]

    return {
        'shape': X.shape,
        'nan_count': nan_count,
        'inf_count': inf_count,
        'constant_columns': constant_columns,
        'near_constant_columns': near_constant_columns,
        'duplicate_columns': duplicate_columns,
        'highly_correlated_pairs': highly_correlated_pairs
    }


NameError: name 'get_project_root' is not defined

In [46]:
import time

while True:
  time.sleep(60)


KeyboardInterrupt: 